<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🧭 M3 Lab 1 · 并行深度研究 Crew（CrewAI CLI 项目）

**一句话**：把 M2 的 4 步串行研究 crew 拆成「主题 / 次要主题」两条线，研究并行跑；顺便学会用 `crewai create crew` 生成的 CLI 项目结构。

<span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🏷️ Lab</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">📄 C1M3_Lab_1_deep_research_p1.ipynb</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">▶️ cd deep_research_crew &amp;&amp; crewai run</span><span style="display:inline-block; background:rgba(15,23,42,0.55); border:1px solid rgba(147,197,253,0.4); color:#e0e7ff; padding:2px 10px; border-radius:999px; font-size:0.82em; margin:2px 6px 2px 0;">🧩 前置：M2 Deep Research Crew · guardrail · knowledge / memory</span>

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**要解决的问题**：做研究顾问，要快速给客户交付**查证过、带引用**的报告。M2 的 crew 一步接一步，研究阶段最慢。
本 lab：
- 规划时把问题拆成 **MAIN**（核心）和 **SECONDARY**（背景）两类主题；
- 两类研究 **同时跑**（`async_execution=True`），各自再做事实核查；
- 不再在 notebook 里写 crew，而是用 **CrewAI CLI 项目**（yaml + `crew.py` + `main.py`），更接近生产。

本 notebook 只是「驱动器」：真正的代码都在 `deep_research_crew/` 里。所有 `START CODE HERE` 空位**你已经填好**（与 `Solution/` 一致，只差空格缩进）。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🧠 从 M2 串行到 M3 并行

</div>

<img src="./Screenshot 2026-10-11 at 22.37.51_enhanced.png" width="420">  <img src="./images/m3-lab1-agents-tasks-diagram.png" width="560">

<div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(260px,1fr)); gap:10px; width:97%;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #f87171; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047;">M2：4 个 task 串行</div><ul style="margin:6px 0 0 0; padding-left:18px; font-size:0.92em; line-height:1.6;"><li style="margin:2px 0;">plan → research → verify → write</li><li style="margin:2px 0;">研究只有一条线，必须等它查完才能核查</li><li style="margin:2px 0;">保存报告靠 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@after_kickoff</code> hook</li></ul></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047;">M3：6 个 task，研究并行</div><ul style="margin:6px 0 0 0; padding-left:18px; font-size:0.92em; line-height:1.6;"><li style="margin:2px 0;">plan → <b>research_main ∥ research_secondary</b> → validate_main → validate_secondary → write</li><li style="margin:2px 0;"><b>agent 不变</b>（还是 4 个），只是同一个 agent 接两个 task</li><li style="margin:2px 0;">保存报告改用 Task 的 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">output_file</code>（不再需要 hook）</li></ul></div></div>

<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">真实执行顺序（Process.sequential）：两个 async 研究同时启动；遇到第一个同步 task 时先等所有 async 结果</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #a78bfa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">create_research_plan</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">research_main_topics ⚡async</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">research_secondary_topics ⚡async</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #60a5fa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">validate_main_topics</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #60a5fa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">validate_secondary_topics</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">write_final_report 🛡️guardrail</span></div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**并行只发生在研究这一步**：`validate_*` 两个都是同步 task，所以核查仍是一前一后（图里画成两条并行线，但代码只把 research 设成 async）。
CrewAI 的规则（`crew.py::_execute_tasks` 源码）：async task 交给线程池立刻返回 future；下一个**同步** task 开始前，会先等**所有**未完成的 future。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🗂️ CLI 项目结构

</div>

这个文件夹是用 `crewai create crew deep_research_crew` 生成的模板，再由课程填了一部分。和前面 notebook 里写 crew 做的是同一件事，只是拆进了固定位置，方便以后部署。

<img src="./images/lab1-folder-structure.png" width="200">

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">📄 pyproject.toml</div><div style="font-size:0.92em; line-height:1.6;">项目依赖（钉死 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">crewai[tools]==1.3.0</code>）+ <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">[project.scripts]</code> 入口：<br><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">run_crew = &quot;deep_research_crew.main:run&quot;</code>。<br><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">crewai run</code> 实际执行 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">uv run run_crew</code>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">⚙️ config/agents.yaml · tasks.yaml</div><div style="font-size:0.92em; line-height:1.6;">agent 的 role/goal/backstory；task 的 description / expected_output / <b>agent</b> / <b>context</b>。<br><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">{user_query}</code> 占位符在 kickoff 时被 inputs 替换。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #a78bfa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧩 crew.py</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@CrewBase</code> 类：<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@agent</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@task</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">@crew</code> 方法把 yaml 变成对象，并加上 yaml 写不了的东西（tools、llm、async、guardrail）。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">▶️ main.py</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">run()</code> 里写 inputs 再 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">kickoff</code>；另有 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">train</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">replay</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">test</code> 入口。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #f87171; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🛡️ guardrails/ · tools/</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">write_report_guardrail</code>（M2 Lab 1 的函数）；自定义工具 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">ChartGeneratorTool</code>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #22d3ee; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">📚 knowledge/user_preference.txt</div><div style="font-size:0.92em; line-height:1.6;">knowledge source 的文件必须放在 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">knowledge/</code> 目录，代码里只写文件名。</div></div></div></div>

先做准备：让 `!crewai` / `uv` 一定能找到，并定义一个读文件的小工具。

In [1]:
import os, sys
from pathlib import Path

# VSCode 启动 kernel 时 PATH 不一定包含 .venv/bin → 手动把本 kernel 的 bin 放最前，`!crewai` 才找得到
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"
# 项目有自己的 .venv（crewai 1.3.0）；去掉外层 VIRTUAL_ENV，免得 uv 打印 "does not match the project environment" 警告
os.environ.pop("VIRTUAL_ENV", None)

PROJ = Path("deep_research_crew")              # notebook 的 cwd = 本 lab 文件夹
SRC = PROJ / "src" / "deep_research_crew"

def show(rel):
    """打印项目里的一个文件"""
    print((SRC / rel).read_text())

# 项目文件树（不含 .venv / __pycache__）
for p in sorted(PROJ.rglob("*")):
    if p.is_file() and not {".venv", "__pycache__"} & set(p.parts) and p.name != ".DS_Store":
        print(p)

deep_research_crew/final_report.md
deep_research_crew/knowledge/user_preference.txt
deep_research_crew/pyproject.toml
deep_research_crew/src/deep_research_crew/__init__.py
deep_research_crew/src/deep_research_crew/config/agents.yaml
deep_research_crew/src/deep_research_crew/config/tasks.yaml
deep_research_crew/src/deep_research_crew/crew.py
deep_research_crew/src/deep_research_crew/guardrails/__init__.py
deep_research_crew/src/deep_research_crew/guardrails/guardrails.py
deep_research_crew/src/deep_research_crew/main.py
deep_research_crew/src/deep_research_crew/tools/__init__.py
deep_research_crew/src/deep_research_crew/tools/chart_generator_tool.py
deep_research_crew/src/deep_research_crew/utils.py
deep_research_crew/uv.lock


In [2]:
# run this cell to avoid warnings
import warnings
warnings.filterwarnings("ignore")

In [3]:
print((PROJ / "pyproject.toml").read_text())

[project]
name = "deep_research_crew"
version = "0.1.0"
description = "deep_research_crew using crewAI"
authors = [{ name = "Your Name", email = "you@example.com" }]
requires-python = ">=3.10,<3.14"
dependencies = [
    "crewai[tools]==1.3.0",
    "exa-py==1.16.1",
    "matplotlib>=3.10.7",
    "pandas>=2.3.3",
    "seaborn>=0.13.2",
]

[project.scripts]
deep_research_crew = "deep_research_crew.main:run"
run_crew = "deep_research_crew.main:run"
train = "deep_research_crew.main:train"
replay = "deep_research_crew.main:replay"
test = "deep_research_crew.main:test"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.crewai]
type = "crew"
project_id = "385692e9-4701-4539-bffb-bec5ec23e059"



<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 1 步：tasks.yaml —— 给每个 task 指定 agent 和 context

</div>

要填两样：
- `agent:` 写 **agents.yaml 里的 key**（= `crew.py` 里 `@agent` 方法名）。main / secondary 共用同一个 agent，例如两个研究 task 都是 `topic_researcher`。
- `context:` 列出「我要读哪些 task 的输出」。它既决定**数据流**，也决定**等待关系**。

下面把 yaml 解析成一张表，比整份读更直观：

In [4]:
import yaml

tasks_cfg = yaml.safe_load((SRC / "config/tasks.yaml").read_text())
for name, cfg in tasks_cfg.items():
    print(f"{name:27} agent={cfg['agent']:17} context={cfg.get('context', [])}")

create_research_plan        agent=research_planner  context=[]
research_main_topics        agent=topic_researcher  context=['create_research_plan']
research_secondary_topics   agent=topic_researcher  context=['create_research_plan']
validate_main_topics        agent=fact_checker      context=['research_main_topics']
validate_secondary_topics   agent=fact_checker      context=['research_secondary_topics']
write_final_report          agent=report_writer     context=['validate_main_topics', 'validate_secondary_topics']


预期：两个研究 task 都依赖 `create_research_plan`；每个 validate 只读对应的那条研究线；`write_final_report` 同时读两个 validate。看一个完整 task 的写法：

In [5]:
print(yaml.dump({"write_final_report": tasks_cfg["write_final_report"]}, sort_keys=False, allow_unicode=True, width=100))

write_final_report:
  description: 'Create a comprehensive report that answers the original query "{user_query}" using all
    validated research data from both MAIN and SECONDARY topic branches. Structure it with clear sections,
    include citations, and provide actionable insights. Use the custom tool to create intuitive charts
    based on the verified information.  Include the generated charts into the final report, any chart
    should be loadded from the deep_research_crew/plots/ path, so you just append the file name after
    that''

    '
  expected_output: 'A final research report containing: 1) Executive summary, 2) Detailed findings that
    answer the user query using main topics, 3) Supporting context from secondary topics, 4) Complete
    source citations, 5) Key insights and recommendations. Must include Summary, Insights/Recommendations,
    and Citations sections. Whenever possible, include the charts inside the report''

    '
  agent: report_writer
  context:
  - v

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**为什么 validate 只给一条线的 context？** context 越少，prompt 越短、越聚焦；核查 main 的 agent 不需要看 secondary 的材料。
`create_research_plan` 没写 context：它是第一个 task，只用 `{user_query}`。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 2 步：guardrails.py —— 复用 M2 的报告检查

</div>

把 M2 Lab 1 的 `write_report_guardrail` 原样复制进 `guardrails/guardrails.py`：报告必须有 Summary、Insights/Recommendations、Citations/References 三个**标题**。返回 `(False, 提示)` 时 CrewAI 把提示发回给 agent 让它重写。

In [6]:
show("guardrails/guardrails.py")

import re
# Copy the guardrail function
### START CODE HERE ###

def write_report_guardrail(output):
    # get the raw output from the TaskOutput object
    try:
        output = output if type(output)==str else output.raw
    except Exception as e:
        return (False, ("Error retrieving the `raw` argument: "
                        f"\n{str(e)}\n"
                        )
                )

    # convert the output to lowercase
    output_lower = output.lower()

    # check that the summary section exists
    if not re.search(r'#+.*summary', output_lower):
        return (False,
                "The report must include a Summary section with a header like '## Summary'"
                )

    # check that the insights or recommendations sections exist
    if not re.search(r'#+.*insights|#+.*recommendations', output_lower):
        return (False,
                "The report must include an Insights section with a header like '## Insights'"
                )

    # check that the cit

不用跑 crew 也能单独测它：把项目的 `src` 加进 `sys.path` 直接 import（kernel 里的 crewai 是 1.15.22，但这个函数只用 `re`）。

In [7]:
sys.path.insert(0, str(SRC.parent))   # 让 `deep_research_crew` 包可被 import
from deep_research_crew.guardrails.guardrails import write_report_guardrail

good = "## Summary\n...\n## Insights\n...\n## Citations\n[1] ..."
bad = "## Summary\n...\n## Citations\n[1] ..."
print(write_report_guardrail(good)[0])   # True → 放行，第二个值是原文
print(write_report_guardrail(bad))       # False + 给 agent 的修改提示

True
(False, "The report must include an Insights section with a header like '## Insights'")


<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 3 步：crew.py —— 工具、async、输出文件、memory

</div>

你在 `crew.py` 里填的 4 处：

| 位置 | 填了什么 | 为什么 |
|---|---|---|
| `topic_researcher` / `fact_checker` | `tools=[EXASearchTool(...), ScrapeWebsiteTool()]` | 研究和核查都要先搜、再读网页原文 |
| `report_writer` | `tools=[ChartGeneratorTool()]` | 用自定义工具从数据画图，存到 `plots/` |
| `research_main_topics` / `research_secondary_topics` | `async_execution=True` | 两条研究线同时跑 |
| `write_final_report` | `guardrails=[write_report_guardrail]`、`markdown_output=True`、`output_file="final_report.md"` | 检查结构 + 自动存文件（替代 M2 的 after-kickoff hook） |
| `crew()` | `memory=True`、`process=Process.sequential` | 开启短期/长期/实体记忆；按 task 定义顺序执行 |

In [8]:
show("crew.py")

import os
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import (
    ScrapeWebsiteTool,
    EXASearchTool
)
# import the guardrail
from deep_research_crew.guardrails.guardrails import write_report_guardrail
# import custom tool
from deep_research_crew.tools.chart_generator_tool import ChartGeneratorTool

from deep_research_crew.utils import get_exa_api_key

# import the knowledge source
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

# set the exa API key
os.environ["EXA_API_KEY"] = get_exa_api_key()

@CrewBase
class ParallelDeepResearchCrew:
    """ParallelDeepResearch crew"""

    # Define the agents
    @agent
    def research_planner(self) -> Agent:
        return Agent(
            config=self.agents_config["research_planner"],
            llm='gpt-4o-mini',
            verbose=True
        )

    @agent
    def topic_researcher(self) -> Agent:

        return Agent(
  

<div style="width:97%;"><div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(230px,1fr)); gap:10px;"><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #a78bfa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🏷️ @CrewBase</div><div style="font-size:0.92em; line-height:1.6;">读取同目录下 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">config/agents.yaml</code>、<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">config/tasks.yaml</code>，放进 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">self.agents_config</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">self.tasks_config</code>。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #fbbf24; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🧑 @agent / @task</div><div style="font-size:0.92em; line-height:1.6;">被装饰的方法会自动收集到 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">self.agents</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">self.tasks</code>。<br><b>task 的顺序 = 方法在类里定义的顺序</b> = sequential 的执行顺序。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #60a5fa; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🔗 yaml 里的字符串</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">agent: topic_researcher</code> 和 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">context: [...]</code> 都按<b>方法名</b>解析成对象，所以 yaml key、方法名必须一致。</div></div><div style="background:#1e293b; border:1px solid #334155; border-top:3px solid #4ade80; border-radius:8px; padding:10px 14px; color:#e2e8f0;"><div style="font-weight:700; color:#fde047; margin-bottom:4px;">🚢 @crew</div><div style="font-size:0.92em; line-height:1.6;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">self.agents</code> / <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">self.tasks</code> 已由装饰器填好，这里只配 process、memory、knowledge 等。</div></div></div></div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🆕 2026-10 版本差异 / 课程小 bug：`markdown_output=True` 根本不是 Task 的参数**。
Task（1.3.0 和 1.15.22 都一样）只有 `markdown` 字段；pydantic 对未知参数**静默忽略**，所以这行什么都没做。想让 CrewAI 在 prompt 里要求 Markdown，应写 `markdown=True`（M3 Lab 2 的 crew 就是这么写的）。
这里报告照样是 Markdown，只因为 guardrail 要求了 `## ` 标题。下面一格直接验证：

</div>

In [9]:
from crewai import Task

t = Task(description="x", expected_output="y", markdown_output=True)   # 不报错……
print("markdown =", t.markdown, "| 有 markdown_output 属性？", hasattr(t, "markdown_output"))  # ……但被丢掉了

markdown = False | 有 markdown_output 属性？ False


<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**其它几个值得注意的参数**
- `EXASearchTool(base_url=os.getenv("EXA_BASE_URL"))`：课程平台走代理；本地没设该变量 → `None` → 用 Exa 官方地址（这个工具允许 `None`）。
- `max_rpm=150`：每分钟最多 150 次请求，防止并行时撞限流；`max_iter=15`：单个 task 最多 15 轮「思考→调工具」。
- `tracing=False`：课程要求别改。
- `os.environ["EXA_API_KEY"] = get_exa_api_key()`：`utils.py` 用 `find_dotenv()` 往上找 `.env`，同时把 OPENAI key 也加载进环境。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🛠️ 第 4 步：main.py —— 写 inputs，然后 kickoff

</div>

唯一要改的是 `run()` 里的 `inputs`：key 必须是 yaml 里的占位符名 `user_query`。

In [10]:
src = (SRC / "main.py").read_text()
print(src[src.index("def run"):src.index("def train")])

def run():
    """
    Run the crew.
    """
    ### START CODE HERE ###
    inputs = {
        "user_query": "Evaluate the top five emerging AI tools for automating competitive market analysis, including their features, limitations, costs, and ideal use cases for a mid-sized marketing firm."
    }
    ### END CODE HERE ###
    ParallelDeepResearchCrew().crew().kickoff(inputs=inputs)





<div style="background:#111827; border:1px dashed #334155; border-radius:8px; padding:10px 16px; width:97%;"><div style="color:#94a3b8; font-weight:700; margin-bottom:6px;">一条命令背后发生了什么</div><span style="display:inline-block; background:#0f172a; border:1.5px solid #a78bfa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">crewai run</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #60a5fa; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;">读 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">pyproject.toml</code>：type = crew</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #fbbf24; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">uv run run_crew</code>（首次自动建 .venv，装 crewai 1.3.0）</span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">main.run()</code></span><span style="color:#fbbf24; margin:0 6px; font-weight:700;">➜</span><span style="display:inline-block; background:#0f172a; border:1.5px solid #4ade80; color:#e2e8f0; padding:4px 12px; border-radius:999px; margin:3px 0; font-size:0.9em;"><code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">ParallelDeepResearchCrew().crew().kickoff(inputs)</code></span></div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:6px 20px; border-radius:6px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 🚀 第 5 步：运行 crew

</div>

`!` 让这行交给系统 shell 执行；`cd deep_research_crew` 进入项目目录，`crewai run` 启动 crew。
⏱️ 整个研究大约几分钟，会真的调用 Exa 搜索和 OpenAI（花钱）；第一次还要先建项目自己的 `.venv`。
> 你最早一次运行在这里报 `SyntaxError: process=Process.None`——CLI 最后只说 `Command '['uv', 'run', 'run_crew']' returned non-zero exit status 1`，**真正的错误在它上面的 traceback 里**。

In [11]:
! cd deep_research_crew && crewai run

🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
│   └── 🧠 Thinking...
├── ✅ Memory Retrieval Completed
🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
│   └── 🧠 Thinking...
├── ✅ Memory Retrieval Completed
🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
│   └── 🧠 Thinking...
├── ✅ Memory Retrieval Completed
└── 🔍 Knowledge Retrieval Started

🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
│   └── 🧠 Thinking...
├── ✅ Memory Retrieval Completed
└── 🔍 Knowledge Retrieval Started

🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
│   └── 🧠 Thinking...
├── ✅ Memory Retrieval Completed
└── 🔍 Knowledge Retrieval Started

🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
│   └── 🧠 Thinking...
├── ✅ Memory Retrieval Completed
└── 🔍 Knowledge Retrieval Started

🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
│   └── 🧠 Thinking...
├── ✅ Memory Retrieval Completed
└── 🔍 Knowledge Retrieval Started

...

🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
│   └── 🧠 Thinking...
├── ✅ Memory Retrieval Completed
└── 🔍 Knowledge Retrieval Started

🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
│   └── 🧠 Thinking...
├── ✅ Memory Retrieval Completed
└── 🔍 Knowledge Retrieval Started

🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
├── ✅ Memory Retrieval Completed
└── 🔍 Knowledge Retrieval Started

🚀 Crew: crew
├── 📋 Task: create_research_plan (ID: abb6e547-01e6-4a85-a136-1bb758ff9406)
│   Status: Executing Task...
├── ✅ Memory Retrieval Completed
└── ✅ Knowledge Retrieval Completed

ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o-mini in organization org-l2CY9uZUbhB9p9qZckRuQpHj on tokens per min (TPM): Limit 2000000, Requested 18450927. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
ERROR:root:OpenAI API call failed: Error code: 429 - {'error': {'message': 'Request too large for gpt-4o-mini in organization org-l2CY9uZUbhB9p9qZckRuQpHj on tokens per min (TPM): Limit 2000000, Requested 18450927. The input or output tokens must be reduced in order to run successfully. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}


<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**▶️ 在终端运行**

也可以在 VSCode 终端里跑（交互输出更顺畅）。

```bash
cd deep_research_crew
crewai run
```

</div>

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关于上面这格的输出（2026-10-21 实测）**
- 输出在中途就**停止刷新**了，但 crew 其实在后台跑完了（约 7 分钟，`final_report.md` 正常生成）。项目钉的 crewai 1.3.0 在 notebook 的 `!` 里长时间运行时会这样；想看完整的 agent 日志就在终端里跑。**结果以下面读出的文件为准。**
- 日志里的 `429 Request too large … Requested 18450927` 是某一次请求塞进了约 1800 万 token（远超每分钟上限），很可能是抓取工具拿回了超大网页；那次调用失败后 crew 继续执行，没有中断。

</div>

报告由 `output_file` 写到 `deep_research_crew/final_report.md`；图表在 `deep_research_crew/plots/`。直接在 notebook 里看（这次运行 report_writer 没有调用画图工具，所以 `plots: 无`——工具只是「可用」，LLM 不一定会用）：

In [12]:
from IPython.display import Markdown, display

report = PROJ / "final_report.md"
print("plots:", sorted(p.name for p in (PROJ / "plots").glob("*.png")) if (PROJ / "plots").exists() else "无")
display(Markdown(report.read_text()))   # 报告里的图片路径写的是 deep_research_crew/plots/…，正好相对本 notebook

plots: 无


**Comprehensive Research Report on Emerging AI Tools for Automating Competitive Market Analysis**

**Executive Summary:**
This report evaluates the top five emerging AI tools designed for automating competitive market analysis tailored for mid-sized marketing firms. The tools assessed are NoimosAI, Crayon, Klue, AlphaSense, and Kompyte. The analysis includes their specific features, limitations, costs, and ideal use cases, providing actionable insights to inform decision-making on AI tool adoption within marketing processes.

---

**1. Identification and Evaluation of AI Tools:**
The emerging AI tools for competitive market analysis identified are:
- **NoimosAI**: Focuses on integrating competitor insights directly into marketing workflows.
- **Crayon**: Specializes in gathering enterprise signals to track competitor activity.
- **Klue**: Provides deal intelligence and battlecards tailored for sales strategies.
- **AlphaSense**: Attracts financial analysts with its extensive research capabilities.
- **Kompyte**: Aids in scaling go-to-market strategies through real-time monitoring.

### Effectiveness and User Satisfaction:
Each tool offers unique benefits depending on the specific needs of marketing teams, with NoimosAI being advantageous for teams with limited resources looking for seamless integration. All tools leverage advanced machine learning capabilities to streamline the analysis of market data.

---

**2. Features of AI Tools:**
Each of the top five AI tools provides distinctive features:
- **NoimosAI**: Offers SEO analytics and strategy support, connecting insights with marketing execution.
- **Crayon**: Enables tracking of competitor activities through comprehensive signal collection.
- **Klue**: Features battlecard creation for enhanced decision-making in sales scenarios.
- **AlphaSense**: Provides access to a wide range of cited research, benefiting financial analysis.
- **Kompyte**: Focuses primarily on monitoring and adjusting go-to-market strategies effectively.

---

**3. Limitations of AI Tools:**
While these tools present significant advantages, they also have inherent limitations:
- **NoimosAI**: Requires human oversight for context validation.
- **Crayon**: May lead to overwhelmed users with data if not properly managed.
- **Klue**: Can have a learning curve for optimizing battlecards effectively.
- **AlphaSense**: Pricing is not transparent and relies on direct sales contact.
- **Kompyte**: Might struggle with catching nuances of competitor engagements without user input.

---

**4. Cost Analysis:**
The pricing structures are as follows:
- **NoimosAI**: Offers a Pro plan at **$99/user/month**.
- **Crayon, Klue, AlphaSense, Kompyte**: Require potential buyers to contact sales for customized pricing options, reflecting a lack of transparency.

### Cost vs. ROI Analysis:
Potential returns are significant, as adopting AI tools may lead to faster insights and data analysis capabilities, aligning with cost savings over time.

---

**5. Ideal Use Cases for Mid-Sized Marketing Firms:**
The following use cases highlight practical applications of the AI tools:
- **Content Creation**: Generate blog drafts and social media posts.
- **Email Marketing**: Optimize email sequences based on audience insights.
- **SEO Optimization**: Utilize AI to enhance website visibility through keyword integration.
- **Social Media Management**: Automate post scheduling and engaging visuals.
- **Performance Reporting**: Streamline analytics through automated report generation.
- **Lead Interaction**: Implement AI chatbots for improved customer engagement.

---

### Summary:
This report concludes that each emerging AI tool provides unique capabilities that can significantly enhance competitive market analysis for mid-sized marketing firms. The effective deployment of these tools can lead to significant insights while ensuring agile and informed marketing strategies.

### Key Insights and Recommendations:
- **Adopt Incrementally**: Begin with one or two tools that align with immediate business needs.
- **Gauge User Feedback**: Continuously evaluate user experience with AI tools to optimize usage.
- **Emphasize Training**: Provide internal training to maximize the potential of AI functionalities.

### Citations:
1. Kaufman Rossin, "State of Artificial Intelligence in the Mid-Market," 2026.
2. King et al., "User Privacy and Large Language Models," 2025.
3. Compttr, "AI Competitive Analysis vs Traditional CI Platforms," 2026.

Incorporating these AI tools into marketing strategies can position firms to stay competitive in dynamic market environments. By evaluating features, costs, and use cases, decision-makers can leverage insights to adopt the most suitable AI solutions to drive their marketing success.

<div class="dark-error" style="background:#2d1f1f; color:#fca5a5; padding:10px 24px; border-left:4px solid #f87171; border-radius:4px; width:97%;"><style>.dark-error strong{color:#fde047;}</style>

**⚠️ 坑**

| 坑 | 现象 | 原因 / 对策 |
|---|---|---|
| `markdown_output=True` | 没报错也没效果 | Task 字段叫 `markdown`；pydantic 静默丢弃未知参数 |
| 把 3 个连续 task 都设成 async 放在末尾 | `The crew must end with at most one asynchronous task` | crew 末尾最多一个 async task；后面要有同步 task 汇总 |
| 以为 validate 也并行了 | 核查阶段时间没缩短 | 只有 `async_execution=True` 的 task 并行；同步 task 会先等所有 async 完成 |
| yaml 的 `agent:` 写成 role 文本（如 `Topic Researcher`） | KeyError | 要写 agents.yaml 的 key / `@agent` 方法名 `topic_researcher` |
| `crewai run` 只显示 `exit status 1` | 看不出哪里错 | 往上翻 traceback；常见是项目代码里还留着 `None` 占位 |
| knowledge 文件路径写成 `knowledge/user_preference.txt` | 找不到文件 | `TextFileKnowledgeSource` 自动在 `knowledge/` 下找，只写文件名 |
| kernel 里 crewai 是 1.15.22，项目里是 1.3.0 | 两边行为可能不同 | `crewai run` 用项目自己的 `.venv`（按 pyproject 钉死的版本） |

</div>

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

**🔁 复习自测**（先自己答，再点开看答案）

<details><summary><b>1. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">research_main_topics</code> 和 <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">research_secondary_topics</code> 为什么能用同一个 agent？</b></summary>

agent 只是「角色 + 工具 + LLM」配置，task 才是具体工作。同一个 `topic_researcher` 可以接多个 task，各自带不同的 description。

</details>

<details><summary><b>2. 两个研究 task 是 async，<code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">validate_main_topics</code> 是同步的，执行时会怎样？</b></summary>

两个研究同时启动（线程池 future）；轮到 `validate_main_topics` 时，crew 先等**所有**未完成的 async task 结束，再把结果作为 context 交给它。

</details>

<details><summary><b>3. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">context</code> 有哪两个作用？</b></summary>

①把指定 task 的输出拼进本 task 的 prompt；②决定依赖关系（要等这些 task 的结果）。

</details>

<details><summary><b>4. M2 用 after-kickoff hook 存报告，这里换成了什么？</b></summary>

`write_final_report` 的 `output_file="final_report.md"`，task 完成后自动把输出写进文件（相对于运行目录 `deep_research_crew/`）。

</details>

<details><summary><b>5. <code style="background:#0b1220; color:#fde68a; padding:1px 5px; border-radius:4px; font-size:0.92em;">crewai run</code> 实际执行了什么？</b></summary>

读 `pyproject.toml`，用 `uv run run_crew` 在项目自己的 `.venv` 里调用 `deep_research_crew.main:run`。

</details>

<details><summary><b>6. 想让 CrewAI 要求输出 Markdown，正确参数是？</b></summary>

`markdown=True`。`markdown_output` 不存在，会被静默忽略。

</details>

</div>

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：**协调模式 = 改 task 的依赖与执行方式**：yaml 里用 `context` 连线，`async_execution=True` 让独立研究并行；CLI 项目把这些拆进 yaml / `crew.py` / `main.py`，一条 `crewai run` 跑起来。

</div>